[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads, max_seq_len=512):
        super().__init__()
        assert num_heads % num_kv_heads == 0, "num_heads must be divisible by num_kv_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_k = d_model // num_heads
        self.groups = num_heads // num_kv_heads

        # Projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k)
        self.W_o = nn.Linear(d_model, d_model)

        # Causal Mask (Lower Triangular)
        self.register_buffer('bias', torch.tril(torch.ones(max_seq_len, max_seq_len))
                                          .view(1, 1, 1, max_seq_len, max_seq_len))

    def forward(self, x):
        B, S, C = x.size()

        # 1. Linear Projections
        q = self.W_q(x) # (B, S, d_model)
        k = self.W_k(x) # (B, S, num_kv_heads * d_k)
        v = self.W_v(x) # (B, S, num_kv_heads * d_k)

        # 2. Reshape for Grouping
        # Split Q into (num_kv_heads, groups_per_head)
        q = q.view(B, S, self.num_kv_heads, self.groups, self.d_k).transpose(1, 2)
        # Result: (B, num_kv_heads, S, groups, d_k) -> swap to (B, num_kv_heads, groups, S, d_k)
        q = q.transpose(2, 3)

        # Reshape K and V to match the KV head count
        k = k.view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2).unsqueeze(2)
        v = v.view(B, S, self.num_kv_heads, self.d_k).transpose(1, 2).unsqueeze(2)
        # Result K/V: (B, num_kv_heads, 1, S, d_k)

        # 3. Scaled Dot-Product Attention (Broadcasting magic happens here)
        # (B, nk, groups, S, d_k) @ (B, nk, 1, d_k, S) -> (B, nk, groups, S, S)
        scaled_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply Mask
        scaled_scores = scaled_scores.masked_fill(self.bias[:, :, :, :S, :S] == 0, float("-inf"))

        attention_weights = F.softmax(scaled_scores, dim=-1)

        # 4. Output Projection
        # (B, nk, groups, S, S) @ (B, nk, 1, S, d_k) -> (B, nk, groups, S, d_k)
        output = attention_weights @ v

        # Merge heads back together
        output = output.transpose(2, 3).contiguous().view(B, S, C)
        return self.W_o(output)




In [12]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [13]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (6.1ms)
  ✅ [2/5] nn.Linear with correct shapes (1.6ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (2.5ms)
  ✅ [4/5] KV heads are shared correctly (2.8ms)
  ✅ [5/5] Gradient flow (3.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (16.8ms total)
  Progress saved. Run status() to see your dashboard.

